# Atividade Prática: Data Warehouse + ETL + API + Python

**Disciplina:** Business Intelligence e Big Data  

**Professor:** Prof. Dr. Bruno Aguilar da Cunha  

**Grupo:** Andreus Vinícius Araújo de Andrade | Arthur do Amaral Barreto | Cauã de Andrade Nogueira | Gabriel Fonseca de Lemos 

**Tema:** Do Dado Bruto à Decisão: Análise Climática e Monitoramento de Qualidade do Ar  

---

In [3]:
# 1. Configuração do Ambiente e Importação de Bibliotecas
import requests
import pandas as pd
import sqlite3
import os

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


## Exemplo Guiado - Dados Climáticos Históricos
Extração e modelagem de dados de temperatura, chuva e vento para Sorocaba, Curitiba e Recife entre janeiro e março de 2025.

In [4]:
URL_CLIMA = 'https://archive-api.open-meteo.com/v1/archive'
DATA_INICIAL_CLIMA = '2025-01-01'
DATA_FINAL_CLIMA = '2025-03-31'

CIDADES_CLIMA = [
    {'cidade': 'Sorocaba', 'estado': 'SP', 'regiao': 'Sudeste', 'latitude': -23.5015, 'longitude': -47.4526},
    {'cidade': 'Curitiba', 'estado': 'PR', 'regiao': 'Sul', 'latitude': -25.4284, 'longitude': -49.2733},
    {'cidade': 'Recife', 'estado': 'PE', 'regiao': 'Nordeste', 'latitude': -8.0476, 'longitude': -34.8770}
]

def extrair_dados_climaticos(local):
    parametros = {
        'latitude': local['latitude'],
        'longitude': local['longitude'],
        'start_date': DATA_INICIAL_CLIMA,
        'end_date': DATA_FINAL_CLIMA,
        'daily': [
            'temperature_2m_mean',
            'temperature_2m_max',
            'temperature_2m_min',
            'precipitation_sum',
            'wind_speed_10m_max'
        ],
        'timezone': 'America/Sao_Paulo'
    }
    resposta = requests.get(URL_CLIMA, params=parametros, timeout=30)
    resposta.raise_for_status()
    dados_json = resposta.json()
    df = pd.DataFrame(dados_json['daily'])
    df['cidade'] = local['cidade']
    df['estado'] = local['estado']
    df['regiao'] = local['regiao']
    df['latitude'] = local['latitude']
    df['longitude'] = local['longitude']
    return df

lista_df = [extrair_dados_climaticos(c) for c in CIDADES_CLIMA]
df_bruto_clima = pd.concat(lista_df, ignore_index=True)

# Camada de Staging
df_bruto_clima.to_csv('staging_clima.csv', index=False, encoding='utf-8-sig')
print(f'Staging clima salvo com sucesso! Registros: {len(df_bruto_clima)}')
df_bruto_clima.head()

Staging clima salvo com sucesso! Registros: 270


,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,cidade,estado,regiao,latitude,longitude
0,2025-01-01,23.6,28.6,19.3,0.4,11.0,Sorocaba,SP,Sudeste,-23.5015,-47.4526
1,2025-01-02,23.5,29.4,19.5,1.7,13.2,Sorocaba,SP,Sudeste,-23.5015,-47.4526
2,2025-01-03,22.7,27.5,19.7,8.7,18.3,Sorocaba,SP,Sudeste,-23.5015,-47.4526
3,2025-01-04,22.2,26.6,19.7,1.9,13.2,Sorocaba,SP,Sudeste,-23.5015,-47.4526
4,2025-01-05,22.7,28.2,17.8,22.4,17.1,Sorocaba,SP,Sudeste,-23.5015,-47.4526


In [5]:
# Transformações e Regras de Negócio - Clima
df_clima = df_bruto_clima.copy()
df_clima = df_clima.rename(columns={
    'time': 'data',
    'temperature_2m_mean': 'temperatura_media_c',
    'temperature_2m_max': 'temperatura_maxima_c',
    'temperature_2m_min': 'temperatura_minima_c',
    'precipitation_sum': 'precipitacao_mm',
    'wind_speed_10m_max': 'vento_maximo_kmh'
})

df_clima['data'] = pd.to_datetime(df_clima['data'])
for col in ['temperatura_media_c', 'temperatura_maxima_c', 'temperatura_minima_c', 'precipitacao_mm', 'vento_maximo_kmh']:
    df_clima[col] = pd.to_numeric(df_clima[col], errors='coerce')

df_clima['precipitacao_mm'] = df_clima['precipitacao_mm'].fillna(0)
df_clima = df_clima.dropna(subset=['data', 'cidade', 'temperatura_media_c', 'temperatura_maxima_c', 'temperatura_minima_c'])
df_clima = df_clima.drop_duplicates(subset=['cidade', 'data'], keep='last')

df_clima['amplitude_termica_c'] = (df_clima['temperatura_maxima_c'] - df_clima['temperatura_minima_c']).round(2)
df_clima['dia_chuvoso'] = (df_clima['precipitacao_mm'] >= 1.0).astype(int)

print(f'Quantidade final tratada: {len(df_clima)}')
df_clima.head()

Quantidade final tratada: 270


,data,temperatura_media_c,temperatura_maxima_c,temperatura_minima_c,precipitacao_mm,vento_maximo_kmh,cidade,estado,regiao,latitude,longitude,amplitude_termica_c,dia_chuvoso
0,2025-01-01,23.6,28.6,19.3,0.4,11.0,Sorocaba,SP,Sudeste,-23.5015,-47.4526,9.3,0
1,2025-01-02,23.5,29.4,19.5,1.7,13.2,Sorocaba,SP,Sudeste,-23.5015,-47.4526,9.9,1
2,2025-01-03,22.7,27.5,19.7,8.7,18.3,Sorocaba,SP,Sudeste,-23.5015,-47.4526,7.8,1
3,2025-01-04,22.2,26.6,19.7,1.9,13.2,Sorocaba,SP,Sudeste,-23.5015,-47.4526,6.9,1
4,2025-01-05,22.7,28.2,17.8,22.4,17.1,Sorocaba,SP,Sudeste,-23.5015,-47.4526,10.4,1


In [6]:
# Construção do Modelo Dimensional e Carga SQLite - Clima
nomes_dias = {0: 'Segunda-feira', 1: 'Terça-feira', 2: 'Quarta-feira', 3: 'Quinta-feira', 4: 'Sexta-feira', 5: 'Sábado', 6: 'Domingo'}

dim_tempo_clima = df_clima[['data']].drop_duplicates().copy()
dim_tempo_clima['tempo_sk'] = dim_tempo_clima['data'].dt.strftime('%Y%m%d').astype(int)
dim_tempo_clima['ano'] = dim_tempo_clima['data'].dt.year
dim_tempo_clima['mes'] = dim_tempo_clima['data'].dt.month
dim_tempo_clima['dia'] = dim_tempo_clima['data'].dt.day
dim_tempo_clima['trimestre'] = dim_tempo_clima['data'].dt.quarter
dim_tempo_clima['dia_semana'] = dim_tempo_clima['data'].dt.dayofweek.map(nomes_dias)
dim_tempo_clima['fim_de_semana'] = (dim_tempo_clima['data'].dt.dayofweek >= 5).astype(int)
dim_tempo_clima = dim_tempo_clima[['tempo_sk', 'data', 'ano', 'mes', 'dia', 'trimestre', 'dia_semana', 'fim_de_semana']]

dim_local_clima = pd.DataFrame(CIDADES_CLIMA)
dim_local_clima.insert(0, 'local_sk', range(1, len(dim_local_clima) + 1))

fato_clima = df_clima.merge(dim_tempo_clima[['tempo_sk', 'data']], on='data', how='left')
fato_clima = fato_clima.merge(dim_local_clima[['local_sk', 'cidade']], on='cidade', how='left')
fato_clima = fato_clima[['tempo_sk', 'local_sk', 'temperatura_media_c', 'temperatura_maxima_c', 'temperatura_minima_c', 'precipitacao_mm', 'vento_maximo_kmh', 'amplitude_termica_c', 'dia_chuvoso']]

conexao_clima = sqlite3.connect('dw_clima.db')
conexao_clima.execute('PRAGMA foreign_keys = ON;')
conexao_clima.executescript('''
DROP TABLE IF EXISTS fato_clima_diario;
DROP TABLE IF EXISTS dim_tempo;
DROP TABLE IF EXISTS dim_local;

CREATE TABLE dim_tempo (
    tempo_sk INTEGER PRIMARY KEY,
    data TEXT NOT NULL UNIQUE,
    ano INTEGER NOT NULL,
    mes INTEGER NOT NULL,
    dia INTEGER NOT NULL,
    trimestre INTEGER NOT NULL,
    dia_semana TEXT NOT NULL,
    fim_de_semana INTEGER NOT NULL
);

CREATE TABLE dim_local (
    local_sk INTEGER PRIMARY KEY,
    cidade TEXT NOT NULL,
    estado TEXT NOT NULL,
    regiao TEXT NOT NULL,
    latitude REAL NOT NULL,
    longitude REAL NOT NULL
);

CREATE TABLE fato_clima_diario (
    tempo_sk INTEGER NOT NULL,
    local_sk INTEGER NOT NULL,
    temperatura_media_c REAL,
    temperatura_maxima_c REAL,
    temperatura_minima_c REAL,
    precipitacao_mm REAL,
    vento_maximo_kmh REAL,
    amplitude_termica_c REAL,
    dia_chuvoso INTEGER,
    PRIMARY KEY (tempo_sk, local_sk),
    FOREIGN KEY (tempo_sk) REFERENCES dim_tempo(tempo_sk),
    FOREIGN KEY (local_sk) REFERENCES dim_local(local_sk)
);
''')

dim_tempo_carga = dim_tempo_clima.copy()
dim_tempo_carga['data'] = dim_tempo_carga['data'].dt.strftime('%Y-%m-%d')
dim_tempo_carga.to_sql('dim_tempo', conexao_clima, if_exists='append', index=False)
dim_local_clima.to_sql('dim_local', conexao_clima, if_exists='append', index=False)
fato_clima.to_sql('fato_clima_diario', conexao_clima, if_exists='append', index=False)
conexao_clima.commit()
print('Data Warehouse dw_clima.db carregado com sucesso!')

Data Warehouse dw_clima.db carregado com sucesso!


In [7]:
# Consultas SQL Analíticas - Parte 1 (Clima)
consulta_clima = '''
SELECT 
    t.ano,
    t.mes,
    l.cidade,
    ROUND(AVG(f.temperatura_media_c), 2) AS temperatura_media,
    ROUND(MAX(f.temperatura_maxima_c), 2) AS maior_temperatura,
    ROUND(SUM(f.precipitacao_mm), 2) AS precipitacao_acumulada,
    SUM(f.dia_chuvoso) AS quantidade_dias_chuvosos
FROM fato_clima_diario AS f
INNER JOIN dim_tempo AS t ON f.tempo_sk = t.tempo_sk
INNER JOIN dim_local AS l ON f.local_sk = l.local_sk
GROUP BY t.ano, t.mes, l.cidade
ORDER BY t.ano, t.mes, precipitacao_acumulada DESC;
'''
res_clima = pd.read_sql_query(consulta_clima, conexao_clima)
display(res_clima)

,ano,mes,cidade,temperatura_media,maior_temperatura,precipitacao_acumulada,quantidade_dias_chuvosos
0,2025,1,Curitiba,20.79,29.6,227.9,26
1,2025,1,Sorocaba,23.65,34.5,205.2,23
2,2025,1,Recife,26.90,31.6,110.2,24
3,2025,2,Curitiba,22.06,30.6,159.1,22
4,2025,2,Sorocaba,25.15,34.3,132.0,15
5,2025,2,Recife,26.80,31.0,111.3,21
6,2025,3,Curitiba,20.96,32.2,111.2,17
7,2025,3,Recife,27.11,31.7,103.2,24
8,2025,3,Sorocaba,24.35,34.5,76.1,17


## Desafio do Grupo - Monitoramento de Qualidade do Ar
Desenvolvimento de um pipeline ETL completo para dados de poluição atmosférica em 4 cidades brasileiras: **Sorocaba (SP), Olinda (PE), Ourinhos (SP) e São Paulo (SP)**.

### Definição do Grão da Tabela Fato:
> **Grão:** *Uma linha para cada cidade em cada dia.*

In [8]:
# Extração de Dados Horários de Qualidade do Ar
URL_AR = 'https://air-quality-api.open-meteo.com/v1/air-quality'
DATA_INICIAL_AR = '2025-07-01'
DATA_FINAL_AR = '2025-07-31'

CIDADES_AR = [
    {'cidade': 'Sorocaba', 'estado': 'SP', 'regiao': 'Sudeste', 'latitude': -23.5015, 'longitude': -47.4526},
    {'cidade': 'Olinda', 'estado': 'PE', 'regiao': 'Nordeste', 'latitude': -8.0089, 'longitude': -34.8553},
    {'cidade': 'Ourinhos', 'estado': 'SP', 'regiao': 'Sudeste', 'latitude': -22.9789, 'longitude': -49.8706},
    {'cidade': 'São Paulo', 'estado': 'SP', 'regiao': 'Sudeste', 'latitude': -23.5505, 'longitude': -46.6333}
]

def extrair_qualidade_ar(local):
    parametros = {
        'latitude': local['latitude'],
        'longitude': local['longitude'],
        'start_date': DATA_INICIAL_AR,
        'end_date': DATA_FINAL_AR,
        'hourly': ['pm10', 'pm2_5', 'nitrogen_dioxide', 'ozone', 'us_aqi'],
        'timezone': 'America/Sao_Paulo'
    }
    r = requests.get(URL_AR, params=parametros, timeout=30)
    r.raise_for_status()
    dados_json = r.json()
    df = pd.DataFrame(dados_json['hourly'])
    df['cidade'] = local['cidade']
    df['estado'] = local['estado']
    df['regiao'] = local['regiao']
    df['latitude'] = local['latitude']
    df['longitude'] = local['longitude']
    return df

dfs_ar = [extrair_qualidade_ar(c) for c in CIDADES_AR]
df_ar_bruto = pd.concat(dfs_ar, ignore_index=True)

# Staging Área
df_ar_bruto.to_csv('staging_qualidade_ar.csv', index=False, encoding='utf-8-sig')
print(f'Staging de qualidade do ar salvo com sucesso! Total de registros horários: {len(df_ar_bruto)}')
df_ar_bruto.head()

Staging de qualidade do ar salvo com sucesso! Total de registros horários: 2976


,time,pm10,pm2_5,nitrogen_dioxide,ozone,us_aqi,cidade,estado,regiao,latitude,longitude
0,2025-07-01T00:00,1.3,1.2,3.9,42.0,37,Sorocaba,SP,Sudeste,-23.5015,-47.4526
1,2025-07-01T01:00,1.2,1.1,3.3,43.0,35,Sorocaba,SP,Sudeste,-23.5015,-47.4526
2,2025-07-01T02:00,1.2,1.1,2.7,44.0,33,Sorocaba,SP,Sudeste,-23.5015,-47.4526
3,2025-07-01T03:00,1.2,1.1,2.2,44.0,32,Sorocaba,SP,Sudeste,-23.5015,-47.4526
4,2025-07-01T04:00,1.0,0.9,2.0,44.0,30,Sorocaba,SP,Sudeste,-23.5015,-47.4526


In [9]:
# Transformação: Agregação da Granularidade de Horária para Diária
df_ar = df_ar_bruto.copy()
df_ar['time'] = pd.to_datetime(df_ar['time'])
df_ar['data'] = df_ar['time'].dt.date

colunas_medidas_ar = ['pm10', 'pm2_5', 'nitrogen_dioxide', 'ozone', 'us_aqi']
for col in colunas_medidas_ar:
    df_ar[col] = pd.to_numeric(df_ar[col], errors='coerce')
    df_ar[col] = df_ar.groupby('cidade')[col].transform(lambda x: x.fillna(x.median()))

# Indicador horário de qualidade insalubre (US AQI > 50)
df_ar['aqi_ruim_hora'] = (df_ar['us_aqi'] > 50).astype(int)

# Agregação diária por cidade e data
df_ar_diario = df_ar.groupby(['cidade', 'estado', 'regiao', 'latitude', 'longitude', 'data']).agg(
    media_pm2_5=('pm2_5', 'mean'),
    maximo_pm2_5=('pm2_5', 'max'),
    media_pm10=('pm10', 'mean'),
    media_ozonio=('ozone', 'mean'),
    media_no2=('nitrogen_dioxide', 'mean'),
    maior_aqi=('us_aqi', 'max'),
    horas_aqi_ruim=('aqi_ruim_hora', 'sum')
).reset_index()

df_ar_diario['media_pm2_5'] = df_ar_diario['media_pm2_5'].round(2)
df_ar_diario['maximo_pm2_5'] = df_ar_diario['maximo_pm2_5'].round(2)
df_ar_diario['media_pm10'] = df_ar_diario['media_pm10'].round(2)
df_ar_diario['media_ozonio'] = df_ar_diario['media_ozonio'].round(2)
df_ar_diario['media_no2'] = df_ar_diario['media_no2'].round(2)
df_ar_diario['dia_qualidade_ruim'] = (df_ar_diario['maior_aqi'] > 50).astype(int)
df_ar_diario['data'] = pd.to_datetime(df_ar_diario['data'])

print(f'Total de registros no grão diário: {len(df_ar_diario)}')
df_ar_diario.head()

Total de registros no grão diário: 124


,cidade,estado,regiao,latitude,longitude,data,media_pm2_5,maximo_pm2_5,media_pm10,media_ozonio,media_no2,maior_aqi,horas_aqi_ruim,dia_qualidade_ruim
0,Olinda,PE,Nordeste,-8.0089,-34.8553,2025-07-01,5.49,6.8,8.75,66.83,2.35,44,0,0
1,Olinda,PE,Nordeste,-8.0089,-34.8553,2025-07-02,6.59,7.2,10.59,73.88,2.09,37,0,0
2,Olinda,PE,Nordeste,-8.0089,-34.8553,2025-07-03,6.72,9.8,10.60,75.50,1.76,37,0,0
3,Olinda,PE,Nordeste,-8.0089,-34.8553,2025-07-04,6.75,8.3,10.81,77.25,1.62,42,0,0
4,Olinda,PE,Nordeste,-8.0089,-34.8553,2025-07-05,6.97,7.4,10.80,77.67,1.85,40,0,0


In [10]:
# Construção do Modelo Dimensional e Carga no SQLite - Qualidade do Ar
dim_tempo_ar = df_ar_diario[['data']].drop_duplicates().copy()
dim_tempo_ar['tempo_sk'] = dim_tempo_ar['data'].dt.strftime('%Y%m%d').astype(int)
dim_tempo_ar['ano'] = dim_tempo_ar['data'].dt.year
dim_tempo_ar['mes'] = dim_tempo_ar['data'].dt.month
dim_tempo_ar['dia'] = dim_tempo_ar['data'].dt.day
dim_tempo_ar['trimestre'] = dim_tempo_ar['data'].dt.quarter
dim_tempo_ar['dia_semana'] = dim_tempo_ar['data'].dt.dayofweek.map(nomes_dias)
dim_tempo_ar['fim_de_semana'] = (dim_tempo_ar['data'].dt.dayofweek >= 5).astype(int)
dim_tempo_ar = dim_tempo_ar[['tempo_sk', 'data', 'ano', 'mes', 'dia', 'trimestre', 'dia_semana', 'fim_de_semana']]

dim_local_ar = pd.DataFrame(CIDADES_AR)
dim_local_ar.insert(0, 'local_sk', range(1, len(dim_local_ar) + 1))

fato_ar = df_ar_diario.merge(dim_tempo_ar[['tempo_sk', 'data']], on='data', how='left')
fato_ar = fato_ar.merge(dim_local_ar[['local_sk', 'cidade']], on='cidade', how='left')
fato_ar = fato_ar[['tempo_sk', 'local_sk', 'media_pm2_5', 'maximo_pm2_5', 'media_pm10', 'media_ozonio', 'media_no2', 'maior_aqi', 'horas_aqi_ruim', 'dia_qualidade_ruim']]

conexao_ar = sqlite3.connect('dw_qualidade_ar.db')
conexao_ar.execute('PRAGMA foreign_keys = ON;')
conexao_ar.executescript('''
DROP TABLE IF EXISTS fato_qualidade_ar_diario;
DROP TABLE IF EXISTS dim_tempo;
DROP TABLE IF EXISTS dim_local;

CREATE TABLE dim_tempo (
    tempo_sk INTEGER PRIMARY KEY,
    data TEXT NOT NULL UNIQUE,
    ano INTEGER NOT NULL,
    mes INTEGER NOT NULL,
    dia INTEGER NOT NULL,
    trimestre INTEGER NOT NULL,
    dia_semana TEXT NOT NULL,
    fim_de_semana INTEGER NOT NULL
);

CREATE TABLE dim_local (
    local_sk INTEGER PRIMARY KEY,
    cidade TEXT NOT NULL,
    estado TEXT NOT NULL,
    regiao TEXT NOT NULL,
    latitude REAL NOT NULL,
    longitude REAL NOT NULL
);

CREATE TABLE fato_qualidade_ar_diario (
    tempo_sk INTEGER NOT NULL,
    local_sk INTEGER NOT NULL,
    media_pm2_5 REAL,
    maximo_pm2_5 REAL,
    media_pm10 REAL,
    media_ozonio REAL,
    media_no2 REAL,
    maior_aqi INTEGER,
    horas_aqi_ruim INTEGER,
    dia_qualidade_ruim INTEGER,
    PRIMARY KEY (tempo_sk, local_sk),
    FOREIGN KEY (tempo_sk) REFERENCES dim_tempo(tempo_sk),
    FOREIGN KEY (local_sk) REFERENCES dim_local(local_sk)
);
''')

dim_tempo_ar_carga = dim_tempo_ar.copy()
dim_tempo_ar_carga['data'] = dim_tempo_ar_carga['data'].dt.strftime('%Y-%m-%d')
dim_tempo_ar_carga.to_sql('dim_tempo', conexao_ar, if_exists='append', index=False)
dim_local_ar.to_sql('dim_local', conexao_ar, if_exists='append', index=False)
fato_ar.to_sql('fato_qualidade_ar_diario', conexao_ar, if_exists='append', index=False)
conexao_ar.commit()

probs = conexao_ar.execute('PRAGMA foreign_key_check;').fetchall()
print(f'Integridade referencial validada no dw_qualidade_ar.db! Problemas: {probs}')

Integridade referencial validada no dw_qualidade_ar.db! Problemas: []


In [11]:
# Consultas SQL Analíticas da Parte 2 (Qualidade do Ar)
print('--- CONSULTA 1: Ranking Geral de Poluição Média e Pico de AQI ---')
c1 = '''
SELECT 
    l.cidade,
    l.estado,
    l.regiao,
    ROUND(AVG(f.media_pm2_5), 2) AS media_pm2_5_periodo,
    ROUND(AVG(f.media_pm10), 2) AS media_pm10_periodo,
    MAX(f.maior_aqi) AS pico_maximo_aqi
FROM fato_qualidade_ar_diario f
JOIN dim_local l ON f.local_sk = l.local_sk
GROUP BY l.cidade, l.estado, l.regiao
ORDER BY media_pm2_5_periodo DESC;
'''
display(pd.read_sql_query(c1, conexao_ar))

print('--- CONSULTA 2: Horas e Dias com Qualidade do Ar Insalubre (US AQI > 50) ---')
c2 = '''
SELECT 
    l.cidade,
    SUM(f.horas_aqi_ruim) AS total_horas_aqi_ruim,
    SUM(f.dia_qualidade_ruim) AS total_dias_aqi_ruim
FROM fato_qualidade_ar_diario f
JOIN dim_local l ON f.local_sk = l.local_sk
GROUP BY l.cidade
ORDER BY total_horas_aqi_ruim DESC;
'''
display(pd.read_sql_query(c2, conexao_ar))

print('--- CONSULTA 3: Comparativo Dias Úteis (0) vs. Finais de Semana (1) ---')
c3 = '''
SELECT 
    l.cidade,
    t.fim_de_semana,
    ROUND(AVG(f.media_pm2_5), 2) AS media_pm2_5,
    ROUND(AVG(f.maior_aqi), 2) AS aqi_medio
FROM fato_qualidade_ar_diario f
JOIN dim_tempo t ON f.tempo_sk = t.tempo_sk
JOIN dim_local l ON f.local_sk = l.local_sk
GROUP BY l.cidade, t.fim_de_semana
ORDER BY l.cidade, t.fim_de_semana;
'''
display(pd.read_sql_query(c3, conexao_ar))

--- CONSULTA 1: Ranking Geral de Poluição Média e Pico de AQI ---


,cidade,estado,regiao,media_pm2_5_periodo,media_pm10_periodo,pico_maximo_aqi
0,São Paulo,SP,Sudeste,26.62,27.31,184
1,Ourinhos,SP,Sudeste,11.04,11.35,78
2,Sorocaba,SP,Sudeste,10.61,11.06,113
3,Olinda,PE,Nordeste,6.22,9.88,44


--- CONSULTA 2: Horas e Dias com Qualidade do Ar Insalubre (US AQI > 50) ---


,cidade,total_horas_aqi_ruim,total_dias_aqi_ruim
0,São Paulo,630,28
1,Sorocaba,453,24
2,Ourinhos,437,27
3,Olinda,0,0


--- CONSULTA 3: Comparativo Dias Úteis (0) vs. Finais de Semana (1) ---


,cidade,fim_de_semana,media_pm2_5,aqi_medio
0,Olinda,0,6.21,38.13
1,Olinda,1,6.27,38.13
2,Ourinhos,0,10.48,60.35
3,Ourinhos,1,12.63,64.38
4,Sorocaba,0,9.56,58.61
5,Sorocaba,1,13.61,71.75
6,São Paulo,0,24.08,91.52
7,São Paulo,1,33.91,131.63
